In [11]:
import re
from collections import defaultdict
import csv
import nltk
from nltk.corpus import stopwords
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import numpy as np
import os

# Load dataset and count words

In [12]:
dataset_path = "../1-dataset/VUAMC_sentences_labeled.csv"

sentences = []

metaphor_counts = defaultdict(int)
literal_counts = defaultdict(int)

with open(dataset_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sentence = row["sentence"]
        label = int(row["label"])
        metaphors = row["metaphors"].split(";") if label == 1 else []
        metaphors = set(m.strip().lower() for m in metaphors)
        sentences.append((sentence, metaphors))

        words = re.findall(r"\w+", sentence.lower())
        for word in words:
            if word in metaphors:
                metaphor_counts[word] += 1
            else:
                literal_counts[word] += 1

# Filter words

In [13]:
top_words = set(word for word in metaphor_counts if metaphor_counts[word] >= 30 and literal_counts[word] >= 30)

nltk.download("stopwords")
stopwords_set = set(stopwords.words("english"))
top_words -= stopwords_set

print(f"Selected words: {top_words}")

Selected words: {'system', 'given', 'get', 'end', 'go', 'way', 'half', 'put', 'see', 'going', 'high', 'power', 'made', 'set', 'give', 'seen', 'make', 'got', 'far', 'take', 'found', 'within', 'place', 'back', 'long', 'whole', 'great', 'part', 'came', 'took', 'bit', 'come', 'next'}


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mattia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Embeddings

In [14]:
# To find the word's token (or tokens)
def get_token_indices(word, all_tokens, tokenizer):
    word_tokens = tokenizer.tokenize(word) # Tokenize the word to handle sub-tokens
    for i in range(len(all_tokens) - len(word_tokens) + 1):
        if all_tokens[i:i+len(word_tokens)] == word_tokens:
            return list(range(i, i+len(word_tokens)))
    return []

In [ ]:
def embeddings(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    top_words_list = []
    embeddings_list = []
    labels_list = []

    for sentence, metaphors in tqdm(sentences):
        words = set(re.findall(r"\w+", sentence.lower()))
        if words & top_words:
            encoded = tokenizer(sentence, return_tensors="pt", truncation=True)
            with torch.no_grad():
                outputs = model(**encoded.to(device))
                hidden_states = outputs.last_hidden_state.squeeze(0)  # [n_tokens, hidden_dim]

            # Convert input_ids to tokens
            all_tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"].squeeze(0))

            for word in top_words:
                token_idxs = get_token_indices(word, all_tokens, tokenizer)
                if token_idxs:
                    embedding = hidden_states[token_idxs].mean(dim=0).cpu() # Sub-token embeddings mean
                    label = 1 if word in metaphors else 0
                    top_words_list.append(word)
                    embeddings_list.append(embedding.numpy())
                    labels_list.append(label)

    os.makedirs(model_name.split("/")[-1], exist_ok=True)
    np.savez(f"{model_name.split("/")[-1]}/top_words_embeddings.npz", words=top_words_list, embeddings=embeddings_list, labels=labels_list)

In [16]:
model_name = "bert-base-uncased"
embeddings(model_name)

100%|██████████| 16202/16202 [03:22<00:00, 80.07it/s] 


In [17]:
model_name = "../6-fine_tuning/bert-base-uncased_ft"
embeddings(model_name)

100%|██████████| 16202/16202 [03:31<00:00, 76.66it/s] 
